In [2]:
import sqlite3
import pandas as pd
from google.colab import drive

# =====================================================================
# TAHAP 1: DATA ACQUISITION
# =====================================================================
drive.mount('/content/gdrive', force_remount=True)
file_path = "/content/gdrive/My Drive/Basis data/DATA.csv"

# Membaca data CSV (Delimiter diatur ke titik koma)
data = pd.read_csv(file_path, sep=';')
print(f"Total baris data : {len(data)}")
print(f"Daftar Kolom     : {list(data.columns)}\n")

# =====================================================================
# TAHAP 2: CREATE DATABASE & TABLE
# =====================================================================
print("=" * 70)
print("CREATE: Menginisialisasi Database dan Tabel SQLite")
print("=" * 70)

# Membuat koneksi ke SQLite
conn = sqlite3.connect("ecommerce_indonesia.db")
cursor = conn.cursor()

# Mentransfer DataFrame ke tabel SQL bernama 'ecommerce'
data.to_sql('ecommerce', conn, if_exists='replace', index=False)
print("✓ Database 'ecommerce_indonesia.db' berhasil terhubung.")
print(f"✓ Tabel 'ecommerce' berhasil di-generate dengan {len(data)} records.\n")

# --- Helper Function untuk Eksekusi Query READ ---
def run_query_and_display(query_name, sql_query, connection):
    print(f"\n{'-' * 70}")
    print(f"QUERY: {query_name}")
    print(f"SQL  :\n{sql_query}")
    try:
        result = pd.read_sql_query(sql_query, connection)
        print("\nOUPUT HASIL:")
        print(result.to_string(index=False))
        return result
    except Exception as e:
        print(f"ERROR: Terjadi kesalahan eksekusi -> {e}")
        return None

# =====================================================================
# TAHAP 3: READ (Eksplorasi & Analisis Data)
# =====================================================================
print("\n" + "=" * 70)
print("READ: Menjalankan Query Analitik Data E-Commerce")
print("=" * 70)

# QUERY 1: Top 5 Total Pembayaran Terbesar
query_1 = """
SELECT order_id, product_categories, `Metode Pembayaran`, `Total Pembayaran`
FROM ecommerce
ORDER BY `Total Pembayaran` DESC
LIMIT 5;
"""
run_query_and_display("Top 5 Total Pembayaran Terbesar", query_1, conn)

# QUERY 2: Jumlah Order per Provinsi
query_2 = """
SELECT Provinsi, COUNT(order_id) AS jumlah_order, SUM(`Total Pembayaran`) AS total_pembayaran_provinsi
FROM ecommerce
GROUP BY Provinsi
ORDER BY jumlah_order DESC
LIMIT 10;
"""
run_query_and_display("Jumlah Order per Provinsi", query_2, conn)

# QUERY 3: Total Penjualan per Kategori
query_3 = """
SELECT product_categories,
       COUNT(order_id) AS jumlah_order,
       SUM(`Total Pembayaran`) AS total_penjualan,
       ROUND(AVG(`Total Pembayaran`), 2) AS rata_rata_penjualan
FROM ecommerce
GROUP BY product_categories
ORDER BY total_penjualan DESC
LIMIT 10;
"""
run_query_and_display("Total Penjualan per Kategori", query_3, conn)

# QUERY 4: Metode Pembayaran Terpopuler
query_4 = """
SELECT `Metode Pembayaran`,
       COUNT(*) AS total_penggunaan,
       SUM(`Total Pembayaran`) AS total_nilai,
       ROUND(AVG(`Total Pembayaran`), 2) AS rata_rata_nilai
FROM ecommerce
GROUP BY `Metode Pembayaran`
ORDER BY total_penggunaan DESC;
"""
run_query_and_display("Metode Pembayaran Terpopuler", query_4, conn)

# =====================================================================
# TAHAP 4: UPDATE DATA
# =====================================================================
print("\n" + "=" * 70)
print("UPDATE: Memodifikasi Rekod Data Existing")
print("=" * 70)

update_sql = """
UPDATE ecommerce
SET `Status Pesanan` = 'Diproses'
WHERE order_id = 'ORD_0000001';
"""
try:
    cursor.execute(update_sql)
    conn.commit()
    print(f"✓ Berhasil! Status pesanan ORD_0000001 diubah menjadi 'Diproses'.")
except Exception as e:
    print(f"ERROR saat UPDATE: {e}")

# =====================================================================
# TAHAP 5: DELETE DATA
# =====================================================================
print("\n" + "=" * 70)
print("DELETE: Menghapus Data Transaksi Batal")
print("=" * 70)

delete_sql = """
DELETE FROM ecommerce
WHERE `Status Pesanan` = 'Batal';
"""
try:
    cursor.execute(delete_sql)
    conn.commit()
    print(f"✓ Berhasil! Data pesanan dengan status 'Batal' dihapus.")
    print(f"  Jumlah baris yang dihapus: {cursor.rowcount} records.")
except Exception as e:
    print(f"ERROR saat DELETE: {e}")

# =====================================================================
# TAHAP 6: PENUTUPAN KONEKSI
# =====================================================================
conn.close()
print("\n" + "=" * 70)
print("TERMINASI: Koneksi Database SQLite ditutup dengan aman.")
print("=" * 70)

Mounted at /content/gdrive
Total baris data : 18868
Daftar Kolom     : ['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Waktu Pesanan Dibuat', 'order_date']

CREATE: Menginisialisasi Database dan Tabel SQLite
✓ Database 'ecommerce_indonesia.db' berhasil terhubung.
✓ Tabel 'ecommerce' berhasil di-generate dengan 18868 records.


READ: Menjalankan Query Analitik Data E-Commerce

----------------------------------------------------------------------
QUERY: Top 5 Total Pembayaran Terbesar
SQL  :

SELECT order_id, product_categories, `Metode Pembayaran`, `Total Pembayaran`
FROM ecommerce
ORDER BY `Total Pembayaran` DESC
LIMIT 5;


OUPUT HASIL:
   order_id product_categories     Metod